# 50 Startup Problem (Lasso Regression)

This notebook solves the 50_Startups regression problem using the following workflow:

1. Load the dataset from a public URL
2. Preprocess the data and one-hot encode the `State` column
3. Split into train/test sets
4. Evaluate Lasso models using different numbers of features (1 to 4)
5. Report the best-performing feature set and MSE


In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error


## Step 1: Load data from URL

In [ ]:
url = "https://raw.githubusercontent.com/harimittapalli/Mulitple-Linear-Reggression/master/50_Startups.csv"
data = pd.read_csv(url)
print(data.head())
print("\nDataset shape:", data.shape)
print(data.info())


## Step 2: Preprocess data and one-hot encode `State`

In [ ]:
# One-hot encode the categorical State variable
data = pd.get_dummies(data, columns=['State'], drop_first=True)

print(data.head())
print("\nColumns after encoding:", list(data.columns))


In [ ]:
X = data.drop(columns=['Profit'])
y = data['Profit']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)


## Step 3: Evaluate Lasso models for different numbers of features

In [ ]:
results = []

# Test 1 to 4 features, selecting the best combination for each size
for num_features in range(1, 5):
    best_mse = float('inf')
    best_features = []

    for features in combinations(X.columns, num_features):
        feature_list = list(features)

        model = Lasso(alpha=1.0)
        model.fit(X_train[feature_list], y_train)

        y_pred = model.predict(X_test[feature_list])
        mse = mean_squared_error(y_test, y_pred)

        if mse < best_mse:
            best_mse = mse
            best_features = feature_list

    results.append({
        'num_features': num_features,
        'features': best_features,
        'mse': best_mse
    })

results_df = pd.DataFrame(results)
print(results_df[['num_features', 'features', 'mse']])


## Step 4: Best model summary

In [ ]:
best_result = results_df.loc[results_df['mse'].idxmin()]

print("Best overall feature set:")
print("Number of features:", best_result['num_features'])
print("Features:", best_result['features'])
print("MSE:", best_result['mse'])


## Step 5: Final prediction and MSE

In [ ]:
final_features = best_result['features']
final_model = Lasso(alpha=1.0)
final_model.fit(X_train[final_features], y_train)

y_pred = final_model.predict(X_test[final_features])
final_mse = mean_squared_error(y_test, y_pred)

print(f"Final test MSE: {final_mse}")
print(f"Selected features: {final_features}")
